# C 단계 빠른 변형 생성: FT5 + PRUNE50

이 노트북은 지정된 A parent 10개만 내려받아 **Fine-tuning 5 epochs**와 **50% weight pruning** descendant 20개를 생성합니다. Adversarial fine-tuning과 quantization은 실행하지 않습니다. 프로젝트와 결과를 Google Drive에 저장하므로 Colab runtime이 종료되어도 다음 세션에서 이어갈 수 있습니다.

In [2]:
# 1. Google Drive를 연결합니다. 최초 실행 시 Google 계정 승인이 필요합니다.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2. Drive에 repository를 한 번만 clone합니다.
# 이미 존재하면 생성된 metadata를 보존하기 위해 자동 git pull을 하지 않습니다.
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/chlwhdduq2357/Fingerprinting-Model-Zoo.git'
REPO = Path('/content/drive/MyDrive/Fingerprinting-Model-Zoo')
DATA_ROOT = Path('/content/drive/MyDrive/Fingerprinting-Model-Zoo-data')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
print('작업 경로:', Path.cwd())

작업 경로: /content/drive/MyDrive/Fingerprinting-Model-Zoo


In [3]:
# 3. Colab에 이미 설치된 CUDA용 PyTorch는 유지하고 프로젝트만 editable install합니다.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', '.'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA 사용 가능:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('런타임 > 런타임 유형 변경에서 GPU를 선택한 뒤 다시 실행하세요.')
print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA 사용 가능: True
GPU: Tesla T4


In [4]:
# 4. C 생성에 필요한 A parent 10개만 선택적으로 다운로드합니다.
# 전체 A/B/B2 90개 archive를 받지 않습니다. 이미 hash가 맞는 파일은 자동으로 건너뜁니다.
PARENTS = ['A001', 'A002', 'A003', 'A006', 'A008', 'A010', 'A013', 'A015', 'A027', 'A029']
download_command = [
    sys.executable, 'scripts/download_models.py',
    '--model-id', *PARENTS, '--workers', '2'
]
subprocess.run(download_command, check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/download_models.py', '--model-id', 'A001', 'A002', 'A003', 'A006', 'A008', 'A010', 'A013', 'A015', 'A027', 'A029', '--workers', '2'], returncode=0)

In [5]:
# 5. 먼저 작은 ResNet20(A002) 하나로 전체 pipeline을 확인합니다.
# 결과는 C005(FT5), C006(PRUNE50)이며 Drive에 즉시 저장됩니다.
pilot_command = [
    sys.executable, 'scripts/generate_c_fast.py',
    '--model-id', 'A002',
    '--transforms', 'ft5', 'prune50',
    '--device', 'cuda', '--amp',
    '--data-root', str(DATA_ROOT),
    '--batch-size', '128', '--workers', '2'
]
subprocess.run(pilot_command, check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/generate_c_fast.py', '--model-id', 'A002', '--transforms', 'ft5', 'prune50', '--device', 'cuda', '--amp', '--data-root', '/content/drive/MyDrive/Fingerprinting-Model-Zoo-data', '--batch-size', '128', '--workers', '2'], returncode=0)

In [6]:
# 6. 나머지 parent를 한 모델씩 처리합니다.
# 세션이 끊기면 1~4번 셀을 다시 실행한 뒤 이 셀을 재실행하세요.
# checkpoint와 config hash가 맞는 완료 모델은 건너뛰므로 처음부터 다시 학습하지 않습니다.
for parent_id in PARENTS:
    print('\n' + '=' * 70)
    print('처리 중:', parent_id)
    command = [
        sys.executable, 'scripts/generate_c_fast.py',
        '--model-id', parent_id,
        '--transforms', 'ft5', 'prune50',
        '--device', 'cuda', '--amp',
        '--data-root', str(DATA_ROOT),
        '--batch-size', '128', '--workers', '2'
    ]
    subprocess.run(command, check=True)


처리 중: A001

처리 중: A002

처리 중: A003

처리 중: A006

처리 중: A008

처리 중: A010

처리 중: A013

처리 중: A015

처리 중: A027

처리 중: A029


In [7]:
# 7. 생성 수, 정확도, lineage를 확인합니다.
import json
rows = json.loads((REPO / 'metadata/models.json').read_text(encoding='utf-8'))
created = [row for row in rows if row.get('group') == 'C' and row.get('transform_type') in {'ft5', 'prune50'}]
for row in sorted(created, key=lambda item: item['model_id']):
    print(
        row['model_id'], row['parent_id'], row['transform_type'],
        f"acc={row['cifar10_test_accuracy_verified']:.2f}%",
        'lineage=' + row['lineage_id']
    )
print(f'\n완료: {len(created)}/20')
assert len(created) == 20

C001 A001 ft5 acc=92.36% lineage=original_001
C002 A001 prune50 acc=90.51% lineage=original_001
C005 A002 ft5 acc=94.05% lineage=original_002
C006 A002 prune50 acc=89.45% lineage=original_002
C009 A003 ft5 acc=95.33% lineage=original_003
C010 A003 prune50 acc=94.00% lineage=original_003
C013 A006 ft5 acc=95.54% lineage=original_006
C014 A006 prune50 acc=93.36% lineage=original_006
C017 A008 ft5 acc=95.90% lineage=original_008
C018 A008 prune50 acc=94.18% lineage=original_008
C021 A010 ft5 acc=96.29% lineage=original_010
C022 A010 prune50 acc=95.80% lineage=original_010
C025 A013 ft5 acc=95.79% lineage=original_013
C026 A013 prune50 acc=94.14% lineage=original_013
C029 A015 ft5 acc=97.06% lineage=original_015
C030 A015 prune50 acc=97.07% lineage=original_015
C033 A027 ft5 acc=93.90% lineage=original_027
C034 A027 prune50 acc=94.04% lineage=original_027
C037 A029 ft5 acc=93.24% lineage=original_029
C038 A029 prune50 acc=93.18% lineage=original_029

완료: 20/20


In [10]:
from pathlib import Path
import os
import site
import subprocess
import sys

REPO = Path("/content/drive/MyDrive/Fingerprinting-Model-Zoo")
os.chdir(REPO)

# Colab에 설치된 CUDA PyTorch는 그대로 두고 프로젝트만 등록
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."],
    check=True,
)

# 현재 notebook kernel에 editable package 경로 즉시 반영
site.addsitedir(site.getsitepackages()[0])

import model_zoo

print("model_zoo import 성공:", model_zoo.__file__)

model_zoo import 성공: /content/drive/MyDrive/Fingerprinting-Model-Zoo/__init__.py


In [11]:
from model_zoo.core import ROOT, models, sha256
from model_zoo import pair_relation

parents = [
    "A001", "A002", "A003", "A006", "A008",
    "A010", "A013", "A015", "A027", "A029",
]

rows = {row["model_id"]: row for row in models()}

for index, parent_id in enumerate(parents):
    parent = rows[parent_id]

    ft_id = f"C{index * 4 + 1:03d}"
    prune_id = f"C{index * 4 + 2:03d}"

    ft = rows[ft_id]
    pruned = rows[prune_id]

    for child in (ft, pruned):
        path = ROOT / child["local_checkpoint_path"]

        assert path.exists()
        assert sha256(path) == child["sha256"]
        assert child["parent_id"] == parent_id
        assert child["lineage_id"] == parent["lineage_id"]
        assert child["state_dict_sha256"] != parent["state_dict_sha256"]
        assert child["verification"]["samples"] == 10_000
        assert child["verification"]["full_test_set"] is True
        assert pair_relation(parent_id, child["model_id"])["same_lineage"] is True

    assert ft["transform_type"] == "ft5"
    assert ft["transform_config"]["epochs"] == 5

    assert pruned["transform_type"] == "prune50"
    assert abs(
        pruned["transform_metrics"]["actual_mask_sparsity"] - 0.5
    ) < 1e-12
    assert pruned["transform_metrics"]["recovery_fine_tuning"] is False

print("C fast-stage audit passed: 20/20")

C fast-stage audit passed: 20/20


In [12]:
# 8. 기존 unified loader로 생성 모델을 실제 호출합니다.
from model_zoo import load_model, pair_relation
images = torch.rand(4, 3, 32, 32, device='cuda')  # normalize하지 않은 [0,1] RGB tensor
ft_model = load_model('C005', device='cuda')      # A002 -> FT5
pruned_model = load_model('C006', device='cuda')  # A002 -> PRUNE50
with torch.inference_mode():
    print('FT logits:', ft_model(images).shape)
    print('Pruned logits:', pruned_model(images).shape)
print('A002-C005:', pair_relation('A002', 'C005'))
print('C005-C006:', pair_relation('C005', 'C006'))

FT logits: torch.Size([4, 10])
Pruned logits: torch.Size([4, 10])
A002-C005: {'same_lineage': True, 'same_architecture': True, 'same_family': True, 'different_architecture': False, 'hard_negative': False}
C005-C006: {'same_lineage': True, 'same_architecture': True, 'same_family': True, 'different_architecture': False, 'hard_negative': False}


===============================================================

In [14]:
from pathlib import Path
import os
import site
import subprocess
import sys

REPO = Path("/content/drive/MyDrive/Fingerprinting-Model-Zoo")
os.chdir(REPO)

# 기존 C metadata와 checkpoint는 유지하면서 최신 코드만 가져옵니다.
subprocess.run(
    ["git", "-C", str(REPO), "pull", "--ff-only"],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."],
    check=True,
)

site.addsitedir(site.getsitepackages()[0])

print("업데이트 완료")

업데이트 완료


In [15]:
DATA_ROOT = Path(
    "/content/drive/MyDrive/Fingerprinting-Model-Zoo-data"
)

PARENTS = [
    "A001", "A002", "A003", "A006", "A008",
    "A010", "A013", "A015", "A027", "A029",
]

for parent_id in PARENTS:
    print("\n" + "=" * 70)
    print("처리 중:", parent_id)

    command = [
        sys.executable,
        "scripts/generate_c_fast.py",
        "--model-id", parent_id,
        "--transforms", "ptq_int8", "prune20",
        "--device", "cuda",
        "--data-root", str(DATA_ROOT),
        "--batch-size", "128",
        "--workers", "2",
    ]

    subprocess.run(command, check=True)


처리 중: A001

처리 중: A002

처리 중: A003

처리 중: A006

처리 중: A008

처리 중: A010

처리 중: A013

처리 중: A015

처리 중: A027

처리 중: A029


In [16]:
import json

rows = json.loads(
    (REPO / "metadata/models.json").read_text(encoding="utf-8")
)

created = sorted(
    [
        row for row in rows
        if row.get("group") == "C"
        and row.get("transform_type")
        in {"ft5", "prune50", "ptq_int8", "prune20"}
    ],
    key=lambda row: row["model_id"],
)

for row in created:
    print(
        row["model_id"],
        row["parent_id"],
        row["transform_type"],
        f"acc={row['cifar10_test_accuracy_verified']:.2f}%",
        f"lineage={row['lineage_id']}",
    )

print(f"\n완료: {len(created)}/40")
assert len(created) == 40

C001 A001 ft5 acc=92.36% lineage=original_001
C002 A001 prune50 acc=90.51% lineage=original_001
C003 A001 ptq_int8 acc=92.47% lineage=original_001
C004 A001 prune20 acc=92.37% lineage=original_001
C005 A002 ft5 acc=94.05% lineage=original_002
C006 A002 prune50 acc=89.45% lineage=original_002
C007 A002 ptq_int8 acc=93.78% lineage=original_002
C008 A002 prune20 acc=93.91% lineage=original_002
C009 A003 ft5 acc=95.33% lineage=original_003
C010 A003 prune50 acc=94.00% lineage=original_003
C011 A003 ptq_int8 acc=95.32% lineage=original_003
C012 A003 prune20 acc=95.23% lineage=original_003
C013 A006 ft5 acc=95.54% lineage=original_006
C014 A006 prune50 acc=93.36% lineage=original_006
C015 A006 ptq_int8 acc=95.03% lineage=original_006
C016 A006 prune20 acc=95.33% lineage=original_006
C017 A008 ft5 acc=95.90% lineage=original_008
C018 A008 prune50 acc=94.18% lineage=original_008
C019 A008 ptq_int8 acc=95.51% lineage=original_008
C020 A008 prune20 acc=95.76% lineage=original_008
C021 A010 ft5 a

In [5]:
from pathlib import Path
import importlib
import os
import site
import subprocess
import sys

REPO = Path("/content/drive/MyDrive/Fingerprinting-Model-Zoo")
os.chdir(REPO)

# 최신 loader 코드 확인
subprocess.run(
    ["git", "-C", str(REPO), "pull", "--ff-only"],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."],
    check=True,
)

site.addsitedir(site.getsitepackages()[0])

# 현재 kernel에 남아 있는 이전 unified loader를 명시적으로 다시 읽습니다.
import model_zoo.loaders.unified as unified

unified = importlib.reload(unified)
load_model = unified.load_model

print("새 loader 위치:", unified.__file__)

새 loader 위치: /content/drive/MyDrive/Fingerprinting-Model-Zoo/loaders/unified.py


In [20]:
from model_zoo.core import ROOT, models, sha256

rows = {
    row["model_id"]: row
    for row in models()
}

c007 = rows["C007"]
checkpoint_path = ROOT / c007["local_checkpoint_path"]

print("checkpoint_format:", c007["checkpoint_format"])
print("loader_spec:", c007["loader_spec"])
print("checkpoint:", checkpoint_path)
print("파일 존재:", checkpoint_path.exists())
print("SHA256 일치:", sha256(checkpoint_path) == c007["sha256"])

assert c007["checkpoint_format"] == "torchscript_int8"
assert c007["loader_spec"]["adapter"] == "local_torchscript_int8"
assert checkpoint_path.exists()
assert sha256(checkpoint_path) == c007["sha256"]

checkpoint_format: torchscript_int8
loader_spec: {'adapter': 'local_torchscript_int8', 'parent': {'adapter': 'vendored', 'module': 'cv.models.resnet_cifar', 'factory': 'resnet20_cifar10'}}
checkpoint: /content/drive/MyDrive/Fingerprinting-Model-Zoo/checkpoints/C/C007_A002_ptq_int8.pt
파일 존재: True
SHA256 일치: True


In [22]:
import torch

images = torch.rand(4, 3, 32, 32)

quantized = load_model(
    "C007",
    device="cpu",
)

with torch.inference_mode():
    logits = quantized(images)

print("출력 shape:", logits.shape)
print("finite:", torch.isfinite(logits).all().item())

assert logits.shape == (4, 10)
assert torch.isfinite(logits).all()

print("C007 PTQ_INT8 load 성공")

KeyError: 'module'

In [23]:
from pathlib import Path
from copy import deepcopy
import json
import os
import uuid

from model_zoo.core import ROOT, models, save_models, sha256, write_json

REPO = Path("/content/drive/MyDrive/Fingerprinting-Model-Zoo")
os.chdir(REPO)

TRANSFORM_SLOT = {
    "ft5": 1,
    "prune50": 2,
    "ptq_int8": 3,
    "prune20": 4,
}

rows = models()
original_rows = deepcopy(rows)
c_rows = [row for row in rows if row.get("group") == "C"]

print("현재 C 모델 수:", len(c_rows))


def desired_c_id(parent_id, transform_type):
    """A 번호에 직접 대응하는 C 번호를 계산합니다."""
    parent_number = int(parent_id[1:])
    slot = TRANSFORM_SLOT[transform_type]
    return f"C{(parent_number - 1) * 4 + slot:03d}"


# old ID → new ID 대응표
id_mapping = {}

for row in c_rows:
    old_id = row["model_id"]
    new_id = desired_c_id(
        row["parent_id"],
        row["transform_type"],
    )
    id_mapping[old_id] = new_id

# 서로 다른 모델이 같은 최종 ID를 요구하는지 검사
desired_ids = list(id_mapping.values())

if len(desired_ids) != len(set(desired_ids)):
    raise RuntimeError("새 C ID가 중복됩니다. metadata를 먼저 점검하세요.")

print("\n변경 예정:")
for old_id, new_id in sorted(id_mapping.items()):
    if old_id != new_id:
        print(f"{old_id} → {new_id}")

# checkpoint, epoch log, verification report를 함께 이동합니다.
move_plan = []

for row in c_rows:
    old_id = row["model_id"]
    new_id = id_mapping[old_id]

    if old_id == new_id:
        continue

    transform = row["transform_type"]
    parent_id = row["parent_id"]

    old_checkpoint = ROOT / row["local_checkpoint_path"]
    new_checkpoint = (
        ROOT
        / "checkpoints"
        / "C"
        / f"{new_id}_{parent_id}_{transform}{old_checkpoint.suffix}"
    )

    if not old_checkpoint.exists():
        raise FileNotFoundError(old_checkpoint)

    move_plan.append((old_checkpoint, new_checkpoint))

    # FT epoch JSONL 등의 실행 로그
    run_directory = ROOT / "runs" / "C"
    if run_directory.exists():
        for old_log in run_directory.glob(f"{old_id}*"):
            new_log = old_log.with_name(
                new_id + old_log.name[len(old_id):]
            )
            move_plan.append((old_log, new_log))

    # verify_models.py가 생성한 개별 결과가 있다면 같이 이동
    verification_directory = ROOT / "reports" / "verification"
    if verification_directory.exists():
        for old_report in verification_directory.glob(f"{old_id}*"):
            new_report = old_report.with_name(
                new_id + old_report.name[len(old_id):]
            )
            move_plan.append((old_report, new_report))

# 중복된 이동 항목 제거
unique_moves = []
seen_moves = set()

for old_path, new_path in move_plan:
    key = (old_path.resolve(), new_path.resolve())
    if key not in seen_moves:
        seen_moves.add(key)
        unique_moves.append((old_path, new_path))

move_plan = unique_moves

# 작업 영역 외부 파일을 건드리지 않는지 검사
root_resolved = ROOT.resolve()

for old_path, new_path in move_plan:
    if not old_path.resolve().is_relative_to(root_resolved):
        raise RuntimeError(f"작업 영역 밖의 파일입니다: {old_path}")

    if not new_path.resolve().is_relative_to(root_resolved):
        raise RuntimeError(f"작업 영역 밖의 파일입니다: {new_path}")

# 새 경로가 재배정 대상이 아닌 기존 파일과 충돌하는지 검사
old_paths = {old.resolve() for old, _ in move_plan}

for _, new_path in move_plan:
    if new_path.exists() and new_path.resolve() not in old_paths:
        raise FileExistsError(
            f"재배정 대상이 아닌 파일과 충돌합니다: {new_path}"
        )

# 1단계: 모든 기존 파일을 충돌하지 않는 임시 이름으로 이동
staged_moves = []

try:
    for old_path, new_path in move_plan:
        token = uuid.uuid4().hex
        temporary_path = old_path.with_name(
            f".renumber-{token}-{old_path.name}"
        )

        old_path.replace(temporary_path)

        staged_moves.append({
            "old": old_path,
            "temporary": temporary_path,
            "new": new_path,
            "finalized": False,
        })

    # 2단계: 임시 파일을 최종 이름으로 이동
    for move in staged_moves:
        move["new"].parent.mkdir(parents=True, exist_ok=True)
        move["temporary"].replace(move["new"])
        move["finalized"] = True

    # metadata 내부 ID와 checkpoint 경로 갱신
    for row in c_rows:
        old_id = row["model_id"]
        new_id = id_mapping[old_id]

        if old_id == new_id:
            continue

        transform = row["transform_type"]
        parent_id = row["parent_id"]

        old_checkpoint = ROOT / row["local_checkpoint_path"]
        new_checkpoint = (
            ROOT
            / "checkpoints"
            / "C"
            / f"{new_id}_{parent_id}_{transform}{old_checkpoint.suffix}"
        )

        row["model_id"] = new_id
        row["original_filename"] = new_checkpoint.name
        row["local_checkpoint_path"] = (
            new_checkpoint.relative_to(ROOT).as_posix()
        )

        if isinstance(row.get("verification"), dict):
            row["verification"]["model_id"] = new_id

        if isinstance(row.get("full_verification"), dict):
            row["full_verification"]["model_id"] = new_id

    # 원본 집단 뒤에 C를 번호순으로 정렬
    non_c_rows = [
        row for row in rows
        if row.get("group") != "C"
    ]

    rows = non_c_rows + sorted(
        c_rows,
        key=lambda row: row["model_id"],
    )

    save_models(rows)

except Exception:
    # 파일 이동 또는 metadata 갱신 실패 시 원래 파일 이름으로 복구
    for move in reversed(staged_moves):
        old_path = move["old"]
        temporary_path = move["temporary"]
        new_path = move["new"]

        if move["finalized"] and new_path.exists():
            new_path.replace(old_path)
        elif temporary_path.exists():
            temporary_path.replace(old_path)

    save_models(original_rows)
    raise

# 이동 후 checkpoint hash 재검사
rows = models()

for row in rows:
    if row.get("group") != "C":
        continue

    checkpoint_path = ROOT / row["local_checkpoint_path"]

    assert checkpoint_path.exists()
    assert sha256(checkpoint_path) == row["sha256"]

    expected_id = desired_c_id(
        row["parent_id"],
        row["transform_type"],
    )

    assert row["model_id"] == expected_id

# 현재 C 요약을 다시 작성
registered = []

for row in rows:
    if row.get("group") == "C":
        registered.append({
            "model_id": row["model_id"],
            "parent_id": row["parent_id"],
            "transform": row["transform_type"],
            "accuracy_percent":
                row.get("cifar10_test_accuracy_verified"),
            "checkpoint_sha256": row.get("sha256"),
            "status": row.get("status"),
        })

registered.sort(key=lambda item: item["model_id"])

write_json(
    ROOT / "reports" / "c_fast_run.json",
    {
        "renumbered": True,
        "registered_count": len(registered),
        "registered_results": registered,
    },
)

print("\nC 번호 재배정 완료")
print("현재 C 모델 수:", len(registered))

for item in registered:
    print(
        item["model_id"],
        item["parent_id"],
        item["transform"],
    )

현재 C 모델 수: 40

변경 예정:
C013 → C021
C014 → C022
C015 → C023
C016 → C024
C017 → C029
C018 → C030
C019 → C031
C020 → C032
C021 → C037
C022 → C038
C023 → C039
C024 → C040
C025 → C049
C026 → C050
C027 → C051
C028 → C052
C029 → C057
C030 → C058
C031 → C059
C032 → C060
C033 → C105
C034 → C106
C035 → C107
C036 → C108
C037 → C113
C038 → C114
C039 → C115
C040 → C116

C 번호 재배정 완료
현재 C 모델 수: 40
C001 A001 ft5
C002 A001 prune50
C003 A001 ptq_int8
C004 A001 prune20
C005 A002 ft5
C006 A002 prune50
C007 A002 ptq_int8
C008 A002 prune20
C009 A003 ft5
C010 A003 prune50
C011 A003 ptq_int8
C012 A003 prune20
C021 A006 ft5
C022 A006 prune50
C023 A006 ptq_int8
C024 A006 prune20
C029 A008 ft5
C030 A008 prune50
C031 A008 ptq_int8
C032 A008 prune20
C037 A010 ft5
C038 A010 prune50
C039 A010 ptq_int8
C040 A010 prune20
C049 A013 ft5
C050 A013 prune50
C051 A013 ptq_int8
C052 A013 prune20
C057 A015 ft5
C058 A015 prune50
C059 A015 ptq_int8
C060 A015 prune20
C105 A027 ft5
C106 A027 prune50
C107 A027 ptq_int8
C108 A027 pr

In [24]:
from pathlib import Path
import importlib.util
import json
import os
import site
import subprocess
import sys
import traceback

REPO = Path("/content/drive/MyDrive/Fingerprinting-Model-Zoo")
DATA_ROOT = Path(
    "/content/drive/MyDrive/Fingerprinting-Model-Zoo-data"
)

os.chdir(REPO)

# 현재 Colab kernel에 editable package 경로 반영
site.addsitedir(site.getsitepackages()[0])

from model_zoo.core import ROOT, models, save_models, sha256
from model_zoo import pair_relation

# metadata에 실제 등록된 A 모델을 번호순으로 가져옵니다.
rows = models()

all_a_ids = sorted(
    [
        row["model_id"]
        for row in rows
        if row.get("group") == "A"
    ],
    key=lambda model_id: int(model_id[1:]),
)

expected_a_ids = [
    f"A{number:03d}"
    for number in range(1, 31)
]

assert all_a_ids == expected_a_ids, (
    f"A001~A030 구성이 아닙니다: {all_a_ids}"
)

required_transforms = {
    "ft5",
    "prune50",
    "ptq_int8",
    "prune20",
}

# 네 변형이 아직 모두 완성되지 않은 parent만 선택합니다.
completed_by_parent = {
    parent_id: set()
    for parent_id in all_a_ids
}

for row in rows:
    if (
        row.get("group") == "C"
        and row.get("parent_id") in completed_by_parent
        and row.get("status") == "verified"
    ):
        completed_by_parent[row["parent_id"]].add(
            row.get("transform_type")
        )

target_parents = [
    parent_id
    for parent_id in all_a_ids
    if completed_by_parent[parent_id] != required_transforms
]

print("전체 A:", len(all_a_ids))
print("추가 처리가 필요한 A:", len(target_parents))
print(target_parents)

# 필요한 parent checkpoint만 선택 다운로드합니다.
# 기존 파일의 hash가 맞으면 downloader가 자동으로 건너뜁니다.
if target_parents:
    subprocess.run(
        [
            sys.executable,
            "scripts/download_models.py",
            "--model-id",
            *target_parents,
            "--workers",
            "2",
        ],
        check=True,
    )

# downloader가 기존 verified parent 상태를 downloaded로 낮춘 경우,
# 같은 SHA256 checkpoint이고 기존 full verification 근거가 있으면 복구합니다.
rows = models()

for row in rows:
    if row.get("group") != "A":
        continue

    checkpoint_path = ROOT / row["local_checkpoint_path"]

    if (
        checkpoint_path.exists()
        and row.get("sha256")
        and sha256(checkpoint_path) == row["sha256"]
        and isinstance(row.get("full_verification"), dict)
        and row["full_verification"].get("passed") is True
    ):
        row["status"] = "verified"

save_models(rows)

# generate_c_fast.py를 파일 수정 없이 현재 kernel으로 불러옵니다.
module_path = REPO / "scripts" / "generate_c_fast.py"

spec = importlib.util.spec_from_file_location(
    "c_stage_generator",
    module_path,
)

generator_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generator_module)

# 실행 중인 module에서만 parent 목록을 A001~A030으로 확장합니다.
# GitHub 파일이나 Drive의 Python 파일은 수정되지 않습니다.
generator_module.PARENTS = all_a_ids

expected_slots = {
    "ft5": 1,
    "prune50": 2,
    "ptq_int8": 3,
    "prune20": 4,
}

assert generator_module.TRANSFORM_SLOT == expected_slots, (
    "generate_c_fast.py가 최신 C 슬롯 구성이 아닙니다."
)

failures = []

# 변형별로 별도 호출하여 한 변형이 실패해도 다음 변형은 계속합니다.
for parent_id in target_parents:
    for transform in [
        "ft5",
        "prune50",
        "ptq_int8",
        "prune20",
    ]:
        print("\n" + "=" * 72)
        print("처리 중:", parent_id, transform)

        arguments = [
            "generate_c_fast.py",
            "--model-id", parent_id,
            "--transforms", transform,
            "--device", "cuda",
            "--data-root", str(DATA_ROOT),
            "--batch-size", "128",
            "--workers", "2",
        ]

        # AMP는 실제 gradient 학습이 있는 FT5에만 적용됩니다.
        if transform == "ft5":
            arguments.append("--amp")

        old_argv = sys.argv

        try:
            sys.argv = arguments
            generator_module.main()

        except (Exception, SystemExit) as error:
            failures.append({
                "parent_id": parent_id,
                "transform": transform,
                "error": repr(error),
            })

            print(
                "실패:",
                parent_id,
                transform,
                repr(error),
            )
            traceback.print_exc()

        finally:
            sys.argv = old_argv

# 최종 120개 검증
rows = models()
by_id = {
    row["model_id"]: row
    for row in rows
}

c_rows = sorted(
    [
        row for row in rows
        if row.get("group") == "C"
    ],
    key=lambda row: row["model_id"],
)

print("\n" + "=" * 72)
print("C 생성 결과:", len(c_rows), "/ 120")
print("이번 실행 실패:", len(failures))

for failure in failures:
    print(failure)

for parent_id in all_a_ids:
    parent_number = int(parent_id[1:])
    parent = by_id[parent_id]

    for transform, slot in expected_slots.items():
        c_id = f"C{(parent_number - 1) * 4 + slot:03d}"

        if c_id not in by_id:
            print("누락:", c_id, parent_id, transform)
            continue

        child = by_id[c_id]
        checkpoint_path = ROOT / child["local_checkpoint_path"]

        assert child["parent_id"] == parent_id
        assert child["transform_type"] == transform
        assert child["lineage_id"] == parent["lineage_id"]
        assert child["status"] == "verified"
        assert checkpoint_path.exists()
        assert sha256(checkpoint_path) == child["sha256"]
        assert child["verification"]["samples"] == 10_000
        assert child["verification"]["full_test_set"] is True
        assert pair_relation(
            parent_id,
            c_id,
        )["same_lineage"] is True

        if transform == "ft5":
            assert child["transform_config"]["epochs"] == 5

        elif transform == "prune50":
            assert abs(
                child["transform_metrics"]["actual_mask_sparsity"]
                - 0.5
            ) < 1e-12

        elif transform == "prune20":
            assert abs(
                child["transform_metrics"]["actual_mask_sparsity"]
                - 0.2
            ) < 1e-12

        elif transform == "ptq_int8":
            assert (
                child["checkpoint_format"]
                == "torchscript_int8"
            )
            assert (
                child["transform_metrics"]
                ["quantized_weight_module_count"]
                > 0
            )
            assert (
                child["transform_metrics"]
                ["activation_quantizer_count"]
                > 0
            )

if not failures and len(c_rows) == 120:
    print("\n전체 C 확장 완료: 120/120")
else:
    print(
        "\n일부 변형이 남았습니다.",
        "같은 셀을 다시 실행하면 완료된 결과는 건너뜁니다.",
    )

전체 A: 30
추가 처리가 필요한 A: 20
['A004', 'A005', 'A007', 'A009', 'A011', 'A012', 'A014', 'A016', 'A017', 'A018', 'A019', 'A020', 'A021', 'A022', 'A023', 'A024', 'A025', 'A026', 'A028', 'A030']

처리 중: A004 ft5
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C013 A004 ft5: start
  epoch 1/5: loss=0.0068, train_acc=99.93%, 42.9s
  epoch 2/5: loss=0.0057, train_acc=99.93%, 42.4s
  epoch 3/5: loss=0.0053, train_acc=99.93%, 42.2s
  epoch 4/5: loss=0.0051, train_acc=99.93%, 42.5s
  epoch 5/5: loss=0.0046, train_acc=99.94%, 42.8s
C013: verified accuracy=96.31%, total=219.3s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A004 prune50
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C014 A004 prune50: start
C014: verified accuracy=95.40%, total=4.3s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A004 ptq_int8
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C015 A004 ptq_int8: start


/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(


C015: verified accuracy=96.02%, total=108.3s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A004 prune20
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C016 A004 prune20: start
C016: verified accuracy=96.19%, total=3.8s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A005 ft5
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C017 A005 ft5: start
  epoch 1/5: loss=0.0829, train_acc=97.80%, 19.0s
  epoch 2/5: loss=0.0791, train_acc=97.78%, 18.1s
  epoch 3/5: loss=0.0755, train_acc=97.93%, 17.7s
  epoch 4/5: loss=0.0720, train_acc=98.03%, 17.4s
  epoch 5/5: loss=0.0714, train_acc=98.02%, 18.0s
C017: verified accuracy=93.52%, total=95.9s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A005 prune50
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C018 A005 prune50: start
C018: verified accuracy=89.81%, total=1.9s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_ru

/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/fx/utils.py:994: UserWarning: QConfig must specify a FixedQParamsObserver or a FixedQParamsFakeQuantize for fixed qparams ops, ignoring QConfig(activation=functools.partial(<class 'torch.ao.quantization.observer.HistogramObserver'>, reduce_range=True){'factory_kwargs': <function _add_module_to_qconfig_obs_ctr.<locals>.get_factory_kwargs_based_on_module_device at 0x7e33d1fd62a0>}, weight=functools.partial(<class 'torch.ao.quantization.observer.PerChannelMinMaxObserver'>, dtype=torch.qint8, qscheme=torch.per_channel_symmetric){'factory_kwargs': <function _add_module_to_qconfig_obs_ctr.<locals>.get_factory_kwargs_based_on_module_device at 0x7e33d1fd62a0>}).
Please use torch.ao.quantization.get_default_qconfig_mapping or torch.ao.quantization.get_default_qat_qconfig_mapping. Example:
    qconfig_mapping = get_default_qconfig_mapping("fbgemm")
    model = prepare_fx(model, qconfig_mapping, example_inputs)
  _activation_post_proce

C027: verified accuracy=93.87%, total=27.1s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A007 prune20
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C028 A007 prune20: start
C028: verified accuracy=93.84%, total=2.0s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A009 ft5
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C033 A009 ft5: start
  epoch 1/5: loss=0.0087, train_acc=99.90%, 36.9s
  epoch 2/5: loss=0.0055, train_acc=99.90%, 35.9s
  epoch 3/5: loss=0.0045, train_acc=99.91%, 35.9s
  epoch 4/5: loss=0.0039, train_acc=99.92%, 36.8s
  epoch 5/5: loss=0.0036, train_acc=99.94%, 36.0s
C033: verified accuracy=95.51%, total=187.9s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A009 prune50
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C034 A009 prune50: start
C034: verified accuracy=92.70%, total=3.0s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_ru

...

/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(


C103: verified accuracy=92.84%, total=44.8s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A026 prune20
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C104 A026 prune20: start
C104: verified accuracy=93.12%, total=2.4s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A028 ft5
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C109 A028 ft5: start
  epoch 1/5: loss=0.0739, train_acc=97.49%, 25.9s
  epoch 2/5: loss=0.0734, train_acc=97.59%, 24.6s
  epoch 3/5: loss=0.0711, train_acc=97.63%, 24.7s
  epoch 4/5: loss=0.0676, train_acc=97.78%, 23.5s
  epoch 5/5: loss=0.0635, train_acc=97.95%, 24.2s
C109: verified accuracy=90.71%, total=128.4s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A028 prune50
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C110 A028 prune50: start
C110: verified accuracy=90.53%, total=2.9s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_ru

/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(


C111: verified accuracy=90.30%, total=26.4s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A028 prune20
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C112 A028 prune20: start
C112: verified accuracy=90.65%, total=2.3s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A030 ft5
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C117 A030 ft5: start
  epoch 1/5: loss=0.0022, train_acc=99.96%, 26.6s
  epoch 2/5: loss=0.0025, train_acc=99.96%, 25.6s
  epoch 3/5: loss=0.0023, train_acc=99.97%, 26.0s
  epoch 4/5: loss=0.0024, train_acc=99.97%, 26.3s
  epoch 5/5: loss=0.0023, train_acc=99.97%, 26.4s
C117: verified accuracy=94.38%, total=138.0s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json

처리 중: A030 prune50
device=cuda, torch=2.11.0+cu128
gpu=Tesla T4
C118 A030 prune50: start
C118: verified accuracy=94.39%, total=5.7s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_ru

AssertionError: 

In [3]:
## 저장상태확인 ##
from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/Fingerprinting-Model-Zoo")
metadata_path = ROOT / "metadata/models.json"

rows = json.loads(metadata_path.read_text(encoding="utf-8"))
c_rows = sorted(
    (row for row in rows if row.get("group") == "C"),
    key=lambda row: row["model_id"],
)

saved = []
missing = []

for row in c_rows:
    checkpoint = ROOT / row["local_checkpoint_path"]

    if checkpoint.is_file():
        saved.append(
            (
                row["model_id"],
                row.get("parent_id"),
                row.get("transform_type"),
                checkpoint.stat().st_size / (1024**2),
                row.get("status"),
            )
        )
    else:
        missing.append(row["model_id"])

print(f"metadata에 등록된 C 모델: {len(c_rows)}개")
print(f"실제 checkpoint 존재:    {len(saved)}개")
print(f"checkpoint 누락:         {len(missing)}개")
print()

for model_id, parent_id, transform, size_mb, status in saved:
    print(
        f"{model_id}  parent={parent_id}  "
        f"transform={transform:9s}  "
        f"size={size_mb:8.2f} MB  status={status}"
    )

if missing:
    print("\n누락된 checkpoint:", missing)

metadata에 등록된 C 모델: 119개
실제 checkpoint 존재:    119개
checkpoint 누락:         0개

C001  parent=A001  transform=ft5        size=    3.70 MB  status=verified
C002  parent=A001  transform=prune50    size=    3.70 MB  status=verified
C003  parent=A001  transform=ptq_int8   size=    0.96 MB  status=verified
C004  parent=A001  transform=prune20    size=    3.70 MB  status=verified
C005  parent=A002  transform=ft5        size=    1.09 MB  status=verified
C006  parent=A002  transform=prune50    size=    1.09 MB  status=verified
C007  parent=A002  transform=ptq_int8   size=    0.31 MB  status=verified
C008  parent=A002  transform=prune20    size=    1.09 MB  status=verified
C009  parent=A003  transform=ft5        size=    3.40 MB  status=verified
C010  parent=A003  transform=prune50    size=    3.41 MB  status=verified
C011  parent=A003  transform=ptq_int8   size=    0.94 MB  status=verified
C012  parent=A003  transform=prune20    size=    3.41 MB  status=verified
C013  parent=A004  transform=ft5  

In [8]:
# ============================================================
# C055 및 PRUNE20 결과 정밀 진단
# 이 셀은 checkpoint나 metadata를 수정하지 않는다.
# ============================================================

import os
import sys
import hashlib
import importlib
from pathlib import Path

import torch

REPO = Path("/content/drive/MyDrive/Fingerprinting-Model-Zoo")
os.chdir(REPO)

import site
site.addsitedir(site.getsitepackages()[0])

# 이전 import 캐시 제거
for module_name in list(sys.modules):
    if module_name == "model_zoo" or module_name.startswith("model_zoo."):
        del sys.modules[module_name]

from model_zoo.core import ROOT, models, sha256
from model_zoo.loaders.unified import load_model

rows = models()
by_id = {row["model_id"]: row for row in rows}

# ------------------------------------------------------------
# 1. A014의 PTQ_INT8 모델인 C055 검사
# ------------------------------------------------------------
c055 = by_id.get("C055")
c055_ok = False

print("=" * 72)
print("C055 진단")
print("=" * 72)

if c055 is None:
    print("실패: metadata에 C055가 없습니다.")
else:
    path = ROOT / c055["local_checkpoint_path"]

    print("parent_id         :", c055.get("parent_id"))
    print("transform_type    :", c055.get("transform_type"))
    print("checkpoint_format :", c055.get("checkpoint_format"))
    print("loader adapter    :", c055.get("loader_spec", {}).get("adapter"))
    print("status            :", c055.get("status"))
    print("accuracy          :", c055.get("cifar10_test_accuracy_verified"))
    print("verification      :", c055.get("verification"))
    print("checkpoint        :", path)
    print("file exists       :", path.is_file())

    identity_ok = (
        c055.get("group") == "C"
        and c055.get("parent_id") == "A014"
        and c055.get("lineage_id") == "original_014"
        and c055.get("transform_type") == "ptq_int8"
        and c055.get("checkpoint_format") == "torchscript_int8"
        and c055.get("loader_spec", {}).get("adapter")
            == "local_torchscript_int8"
        and c055.get("status") == "verified"
    )

    verification = c055.get("verification") or {}
    verification_ok = (
        verification.get("passed") is True
        and verification.get("samples") == 10_000
        and verification.get("full_test_set") is True
    )

    hash_ok = False
    torchscript_ok = False
    unified_loader_ok = False

    if path.is_file():
        actual_hash = sha256(path)
        hash_ok = actual_hash == c055.get("sha256")

        print("SHA256 일치       :", hash_ok)

        try:
            raw = torch.jit.load(str(path), map_location="cpu").eval()

            with torch.inference_mode():
                raw_logits = raw(torch.randn(2, 3, 32, 32))

            torchscript_ok = (
                raw_logits.shape == (2, 10)
                and raw_logits.dtype == torch.float32
                and torch.isfinite(raw_logits).all().item()
            )
        except Exception as exc:
            print("TorchScript 오류  :", repr(exc))

        try:
            model = load_model("C055", device="cpu")

            with torch.inference_mode():
                logits = model(torch.rand(2, 3, 32, 32))

            unified_loader_ok = (
                logits.shape == (2, 10)
                and torch.isfinite(logits).all().item()
            )
        except Exception as exc:
            print("Unified loader 오류:", repr(exc))

    c055_ok = all([
        identity_ok,
        verification_ok,
        hash_ok,
        torchscript_ok,
        unified_loader_ok,
    ])

    print("identity 정상     :", identity_ok)
    print("10,000개 검증     :", verification_ok)
    print("TorchScript 정상  :", torchscript_ok)
    print("Unified loader    :", unified_loader_ok)

print()
print("C055 최종 판정:", "정상" if c055_ok else "재생성 필요")

# ------------------------------------------------------------
# 2. 모든 PRUNE20 모델의 희소율 검사
# ------------------------------------------------------------
print()
print("=" * 72)
print("PRUNE20 진단")
print("=" * 72)

prune20_rows = sorted(
    [
        row for row in rows
        if row.get("group") == "C"
        and row.get("transform_type") == "prune20"
    ],
    key=lambda row: row["model_id"],
)

invalid_prune20 = []
rounding_differences = []

for row in prune20_rows:
    metrics = row.get("transform_metrics") or {}

    eligible = metrics.get("eligible_weight_elements")
    masked = metrics.get("masked_weight_elements")
    actual = metrics.get("actual_mask_sparsity")

    if eligible is None or masked is None or actual is None:
        invalid_prune20.append((row["model_id"], "metadata 누락"))
        continue

    eligible = int(eligible)
    masked = int(masked)
    actual = float(actual)

    # PyTorch pruning은 제거할 파라미터 개수를 정수로 결정한다.
    expected_masked = round(eligible * 0.2)
    recomputed = masked / eligible

    count_ok = abs(masked - expected_masked) <= 1
    ratio_ok = abs(actual - recomputed) < 1e-12

    if not count_ok or not ratio_ok:
        invalid_prune20.append(
            (
                row["model_id"],
                {
                    "eligible": eligible,
                    "masked": masked,
                    "expected_masked": expected_masked,
                    "actual": actual,
                    "recomputed": recomputed,
                },
            )
        )

    rounding_differences.append(
        (
            row["model_id"],
            actual,
            (actual - 0.2) * 100,  # percentage point 차이
        )
    )

print("PRUNE20 등록 모델:", len(prune20_rows))
print("실제 오류 모델    :", len(invalid_prune20))

if invalid_prune20:
    for item in invalid_prune20:
        print("오류:", item)
else:
    print("모든 PRUNE20 모델의 제거 파라미터 수가 정상입니다.")

print()
print("0.2와 차이가 가장 큰 모델 5개:")

for model_id, actual, delta_pp in sorted(
    rounding_differences,
    key=lambda item: abs(item[2]),
    reverse=True,
)[:5]:
    print(
        f"{model_id}: actual={actual:.12f}, "
        f"0.2 대비 차이={delta_pp:+.9f} percentage points"
    )

C055 진단
parent_id         : A014
transform_type    : ptq_int8
checkpoint_format : torchscript_int8
loader adapter    : local_torchscript_int8
status            : verified
accuracy          : 94.24
verification      : {'model_id': 'C055', 'passed': True, 'samples': 10000, 'correct': 9424, 'accuracy_percent': 94.24, 'full_test_set': True, 'preprocessing': 'native', 'device': 'cpu', 'torch_version': '2.11.0+cu128', 'seconds': 91.938, 'date': '2026-09-13T10:54:23Z'}
checkpoint        : /content/drive/MyDrive/Fingerprinting-Model-Zoo/checkpoints/C/C055_A014_ptq_int8.pt
file exists       : True
SHA256 일치       : True
identity 정상     : True
10,000개 검증     : True
TorchScript 정상  : True
Unified loader    : True

C055 최종 판정: 정상

PRUNE20 진단
PRUNE20 등록 모델: 30
실제 오류 모델    : 0
모든 PRUNE20 모델의 제거 파라미터 수가 정상입니다.

0.2와 차이가 가장 큰 모델 5개:
C028: actual=0.199998534326, 0.2 대비 차이=-0.000146567 percentage points
C008: actual=0.199999261709, 0.2 대비 차이=-0.000073829 percentage points
C020: actual=0.199999261709, 0.

In [7]:
# ============================================================
# C055가 실제로 잘못된 경우에만 실행
# A014의 XConv2d mask를 weight에 흡수한 뒤 PTQ_INT8 재생성
# ============================================================

import os
import sys
import copy
import shutil
import importlib.util
from pathlib import Path
from datetime import datetime, timezone

import torch
from torch import nn

REPO = Path("/content/drive/MyDrive/Fingerprinting-Model-Zoo")
os.chdir(REPO)

import site
site.addsitedir(site.getsitepackages()[0])

for module_name in list(sys.modules):
    if module_name == "model_zoo" or module_name.startswith("model_zoo."):
        del sys.modules[module_name]

from model_zoo.core import ROOT, models
from model_zoo.vendor.cv.models.xdensenet import XConv2d

# generate_c_fast.py를 파일 수정 없이 메모리상에서 불러온다.
script_path = ROOT / "scripts/generate_c_fast.py"
spec = importlib.util.spec_from_file_location(
    "c055_recovery_generator",
    script_path,
)
generator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generator)

# A001~A030 전역 번호 체계를 사용해야 A014가 C055가 된다.
generator.PARENTS = tuple(f"A{i:03d}" for i in range(1, 31))

expected_path = (
    ROOT / "checkpoints/C/C055_A014_ptq_int8.pt"
)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
backup_dir = ROOT / "checkpoints/C" / f"c055_backup_{timestamp}"
backup_dir.mkdir(parents=True, exist_ok=True)

metadata_json = ROOT / "metadata/models.json"
metadata_csv = ROOT / "metadata/models.csv"

metadata_json_backup = backup_dir / "models.json"
metadata_csv_backup = backup_dir / "models.csv"

shutil.copy2(metadata_json, metadata_json_backup)
shutil.copy2(metadata_csv, metadata_csv_backup)

checkpoint_backup = None

if expected_path.exists():
    checkpoint_backup = backup_dir / expected_path.name
    shutil.copy2(expected_path, checkpoint_backup)

def replace_xconv_with_equivalent_conv(module):
    """XConv2d의 고정 mask를 weight에 흡수해 일반 Conv2d로 바꾼다."""
    replaced = 0

    for child_name, child in list(module.named_children()):
        if isinstance(child, XConv2d):
            new_conv = nn.Conv2d(
                in_channels=child.in_channels,
                out_channels=child.out_channels,
                kernel_size=child.kernel_size,
                stride=child.stride,
                padding=child.padding,
                dilation=child.dilation,
                groups=child.groups,
                bias=child.bias is not None,
                padding_mode=child.padding_mode,
                device=child.weight.device,
                dtype=child.weight.dtype,
            )

            with torch.no_grad():
                # 기존 XConv2d.forward가 실행하던 weight * mask를
                # 변환 전에 미리 계산하여 일반 convolution weight로 만든다.
                new_conv.weight.copy_(child.weight * child.mask)

                if child.bias is not None:
                    new_conv.bias.copy_(child.bias)

            new_conv.eval()
            setattr(module, child_name, new_conv)
            replaced += 1
        else:
            replaced += replace_xconv_with_equivalent_conv(child)

    return replaced

def make_a014_ptq(parent_id, calibration_loader):
    assert parent_id == "A014"

    parent_model = generator.load_model(
        parent_id,
        device="cpu",
        preprocessing="native",
    )

    network = parent_model.network.cpu().eval()
    probe = torch.randn(3, 3, 32, 32)

    with torch.inference_mode():
        before = network(probe)

    replaced = replace_xconv_with_equivalent_conv(network)

    if replaced == 0:
        raise RuntimeError("A014에서 XConv2d를 찾지 못했습니다.")

    with torch.inference_mode():
        after = network(probe)

    max_error = float((before - after).abs().max())

    if not torch.allclose(before, after, rtol=1e-5, atol=1e-6):
        raise RuntimeError(
            f"XConv2d 변환 전후 출력이 다릅니다. max_error={max_error}"
        )

    scripted, details = generator.quantize_network_static(
        network,
        calibration_loader,
        parent_model.mean,
        parent_model.std,
    )

    details["pre_quantization_rewrite"] = (
        "Fold fixed XConv2d mask into standard Conv2d weight"
    )
    details["xconv_modules_replaced"] = replaced
    details["rewrite_max_abs_error"] = max_error

    return scripted, details

generator.make_ptq_int8 = make_a014_ptq

# 기존 C055가 있어도 이번 복구 작업에서는 반드시 다시 생성한다.
original_completed_output = generator.completed_output

def force_c055_regeneration(row, path, expected_config):
    if path.name.startswith("C055_"):
        return False
    return original_completed_output(row, path, expected_config)

generator.completed_output = force_c055_regeneration

old_argv = sys.argv[:]

try:
    sys.argv = [
        str(script_path),
        "--model-id", "A014",
        "--transforms", "ptq_int8",
        "--device", "cpu",
        "--batch-size", "128",
        "--workers", "2",
    ]

    generator.main()

except Exception:
    # 복구 도중 실패하면 기존 checkpoint와 metadata를 돌려놓는다.
    shutil.copy2(metadata_json_backup, metadata_json)
    shutil.copy2(metadata_csv_backup, metadata_csv)

    if checkpoint_backup is not None:
        shutil.copy2(checkpoint_backup, expected_path)
    elif expected_path.exists():
        expected_path.unlink()

    raise

finally:
    sys.argv = old_argv

print("C055 재생성 완료")
print("백업 디렉터리:", backup_dir)

device=cpu, torch=2.11.0+cu128


100%|██████████| 170M/170M [00:10<00:00, 16.0MB/s]


C055 A014 ptq_int8: start


/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(


C055: verified accuracy=94.24%, total=149.1s
summary=/content/drive/MyDrive/Fingerprinting-Model-Zoo/reports/c_fast_run.json
C055 재생성 완료
백업 디렉터리: /content/drive/MyDrive/Fingerprinting-Model-Zoo/checkpoints/C/c055_backup_20260913T105134Z


## 결과 위치

- Checkpoint: `checkpoints/C/`
- Epoch log: `runs/C/`
- Metadata: `metadata/models.json`, `metadata/models.csv`
- 마지막 실행 요약: `reports/c_fast_run.json`

Checkpoint는 `.gitignore` 대상입니다. 생성된 weight를 일반 Git commit에 추가하지 마세요. A015에서 CUDA OOM이 발생할 때만 5번/6번 셀의 `--batch-size 128`을 `64`로 바꾸고 해당 parent를 다시 실행하세요.